# 02.2 Model Templates: Exercises

## Contents

**Tools:** CADET-GUI widgets: instrument, configuration and simulation; CADET-Process (Python).

**Python used:** Copied model code, parameter changes, model checks and unit connections.

**Practises:** experiment templates, code export, parameter checks and Python model construction.

1. Choose the experiment
2. Export and change the model
3. Complete a Python model

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

Choose a laboratory experiment, export its code, then complete a model in Python.

## Starter instrument

```{note}
The supplied instrument has only its 10 × 2 cm column active: Tracer, porosity 0.4, dispersion `1e-7` m²/s and flow 6 mL/min. Binding is disabled by a zero adsorption rate.
```

```{tip}
Run the cell below once with **Shift+Enter** to create the instrument panels and load the supplied instrument settings for this exercise.
```

In [ ]:
# Show plots inside the notebook; load the shared setup, create the linked panels and apply the tracer settings.
%matplotlib inline
%run ../resources/first_steps/lesson_setup.py
instrument, configuration, solution = instrument_widgets()
populate_instrument(configuration)

```{tip}
Run the next cell with **Shift+Enter** to open the instrument, configuration and simulation panels. Select **Breakthrough** for the supplied tracer settings and click **Run simulation**. The template sets the experiment; the column and binding settings describe the same instrument.
```

In [ ]:
instrument.display()
configuration.display()
solution.display()

## Exercise 1: choose the experiment

Which template would you choose for each measurement?

| Measurement | Laboratory method |
|---|---|
| Accessible column volume | continuous tracer feeding |
| Spreading of a pulse | a short tracer injection |
| Protein elution | load, wash and a salt gradient |

```{dropdown} Answer
Choose Breakthrough, Pulse Injection and LWE respectively.
```

## Exercise 2: export and change the code

1. Select Breakthrough, feed 1 mol/m³ and duration 600 s. Run the GUI simulation.
2. Click **Show code** in the configuration panel, paste the code into the answer cell and run it. Then run the supplied simulation cell and compare the curves.
3. In the copied `Breakthrough(...)` call, change `flow_rate` from `1e-7` to `2e-7` m³/s and run the cell again. Before simulating, estimate the new breakthrough time with $t_R=\varepsilon_tV/Q$ from First Model. Then run the simulation cell again to check your estimate.

`LCFlowSheet(...)` builds the instrument. `Breakthrough(...)` sets its feed, flow and duration.

In [ ]:
# Paste the code from Show code in the configuration panel


In [ ]:
from CADETProcess.simulator import Cadet
simulation_results = Cadet().simulate(process)
_ = simulation_results.solution.outlet.outlet.plot()

```{dropdown} Answer
GUI and exported code reproduce the same curve. Doubling the flow from 6 to 12 mL/min halves the column residence time, from about 126 to 63 s.
```

## Exercise 3: complete a Python model

```{note}
This partly configured tracer model has an inlet, column and outlet, but no connections or process yet. Work only in Python.
```

Run the starter and inspect `missing_parameters`. Complete the model using:

| Setting | Value |
|---|---:|
| Feed concentration | 1 mol/m³ |
| Flow rate | `1e-7` m³/s (6 mL/min) |
| Column length | 0.1 m |
| Column diameter | 0.02 m |
| Porosity | 0.4 |
| Axial dispersion | `1e-7` m²/s |
| Cycle time | 600 s |

Some values are supplied already. Complete the model in three steps below: set the remaining values, connect the units and create the process.

In [ ]:
from CADETProcess.processModel import (
    ComponentSystem, Inlet, LumpedRateModelWithoutPores, Outlet, FlowSheet, Process,
)

exercise_component_system = ComponentSystem(["Tracer"])
exercise_inlet = Inlet(exercise_component_system, "inlet")
exercise_inlet.c = [1]
exercise_column = LumpedRateModelWithoutPores(exercise_component_system, "column")
exercise_column.length = 0.1
exercise_column.diameter = 0.02
exercise_outlet = Outlet(exercise_component_system, "outlet")

print("inlet missing:", exercise_inlet.missing_parameters)
print("column missing:", exercise_column.missing_parameters)
print("outlet missing:", exercise_outlet.missing_parameters)

```{tip}
Use Part 3 of the lesson as a code reference. Check the supplied values too: an empty missing-parameter list only means all required values are set.
```

**Step 1:** set the remaining values of `exercise_inlet` and `exercise_column` from the table.

In [ ]:
# Step 1: set the remaining values


Check that no required parameters are missing.

In [ ]:
print("inlet missing:", exercise_inlet.missing_parameters)
print("column missing:", exercise_column.missing_parameters)
print("outlet missing:", exercise_outlet.missing_parameters)

**Step 2:** create a flow sheet named `exercise_flow_sheet`. Add the three units with `add_unit()` and connect inlet → column → outlet with `add_connection()`.

In [ ]:
# Step 2: flow sheet with connections


**Step 3:** create a `Process` from `exercise_flow_sheet`, store it as `completed_tracer`, and set its cycle time to **600 s**.

In [ ]:
# Step 3: process setup


The breakthrough of the completed model should match the original 6 mL/min tracer experiment.

In [ ]:
from CADETProcess.simulator import Cadet
completed_results = Cadet().simulate(completed_tracer)
_ = completed_results.solution.column.outlet.plot()